# Dictionary Learning: K-SVD  (Aharon et al., 2006)

* Install: pip install ksvd
* Source:  https://github.com/nel215/ksvd
* Paper:   Aharon, Elad, Bruckstein — "K-SVD: An algorithm for designing overcomplete dictionaries for sparse representation", IEEE TSP 2006.

* Algorithm:
*   Step 1 — Sparse Coding: for each signal y_i, solve
    * min ||y_i - D*x_i||  s.t. ||x_i||_0 <= k  using OMP
*   Step 2 — Dictionary Update: for each atom d_j:
    1. Find all signals that use atom j: omega_j = {i : x_j_i != 0}
    2. Compute residual: E_j = Y[:,omega_j] - sum_{l!=j} d_l * x_l[omega_j]
    3. Apply SVD to E_j -> take 1st left singular vector as new d_j and update the coefficients x_j[omega_j] accordingly
*   Repeat until convergence.

* Key difference from MiniBatch:
    *   MiniBatch updates ALL atoms together using gradient on a mini-batch (approximate).
    *   K-SVD updates ONE atom at a time using exact SVD (optimal for that atom given others).

In [2]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../')
 
import os
import numpy as np
from sklearn.model_selection import train_test_split
from src.dictionary import train_ksvd_dictionary, extract_sparse_codes

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
load_dir = '../data/preprocessed/DB5/'

## 1. Loading data slices

In [4]:
print("Carregando tensores de janelas...")
X_windows = np.load(os.path.join(load_dir, 'S01_X_windows.npy'))
y_windows = np.load(os.path.join(load_dir, 'S01_y_windows.npy'))

X_train_raw, X_test_raw, _, _ = train_test_split(
    X_windows, y_windows, test_size=0.2, random_state=42, stratify=y_windows
)

Carregando tensores de janelas...


## 2. K-SVD training

In [6]:
print("=== Treinando o K-SVD Supercompleto ===")
dict_learner, dictionary_atoms = train_ksvd_dictionary(
    X_train_raw, 
    n_components=800, 
    max_iter=100
)

=== Treinando o K-SVD Supercompleto ===
Treinando Approximate K-SVD (800 átomos, 5 coeficientes ativos)...


## 3. Sparse Codes Extraction via OMP

In [7]:
print("\n=== Extraindo Códigos Esparsos ===")
X_train_sparse_ksvd = extract_sparse_codes(X_train_raw, dict_learner)
X_test_sparse_ksvd = extract_sparse_codes(X_test_raw, dict_learner)


=== Extraindo Códigos Esparsos ===


## 4. Saving Matrices

In [8]:
print("Salvando representações esparsas do K-SVD...")
np.save(os.path.join(load_dir, 'S01_X_train_sparse_ksvd.npy'), X_train_sparse_ksvd)
np.save(os.path.join(load_dir, 'S01_X_test_sparse_ksvd.npy'), X_test_sparse_ksvd)
print("K-SVD concluído com sucesso!")

Salvando representações esparsas do K-SVD...
K-SVD concluído com sucesso!
